# Emoji Extractor — Usage Examples

This notebook demonstrates the main features of the `emoji_extractor` package.

Install with: `pip install emoji_extractor`

## 1. Quick Start — Convenience Functions

The simplest way to use the package is with the top-level convenience functions.
These use the latest Unicode version automatically.

In [ ]:
from emoji_extractor import detect_emoji, count_emoji

# Detect whether a string contains any emoji
print(detect_emoji("Hello 👋"))    # True
print(detect_emoji("No emoji"))    # False

In [ ]:
# Count emoji in a single string — returns a collections.Counter
counts = count_emoji("I love 🍎 and 🍌🍌")
print(counts)
print(f"Bananas: {counts['🍌']}")

## 2. The Extractor Class

For more control (version selection, parallelism settings), use the `Extractor` class directly.

In [ ]:
from emoji_extractor import Extractor

ext = Extractor()  # Uses latest Unicode version (17.0)
print(f"Version: {ext.version}")

# Single string counting
result = ext.count_emoji("Great job 🎉🎉🎉")
print(result)

## 3. Single Strings vs Bulk Processing

The package provides two tiers of counting methods:

| Method | Input | Auto-parallel? |
|---|---|---|
| `count_emoji(string)` | One string | No (already ~9µs) |
| `count_all_emoji(iterable)` | List of strings | **Yes**, for ≥1000 lines |

For processing large datasets, `count_all_emoji` automatically distributes work across CPU cores.

In [ ]:
# Counting across many strings at once
sample_texts = [
    "Love this app 🍎",
    "So funny 😂😂",
    "Hello world",
    "Party time 🎉🎊🎉",
    "No emoji here",
    "Hearts 💕❤️💕",
]

totals = ext.count_all_emoji(sample_texts)
print("All emoji across all strings:")
print(totals.most_common())

## 4. Tone-Modifiable Emoji (TME)

Detect and count emoji that support skin tone modifiers (`🏻🏼🏽🏾🏿`).

In [ ]:
# count_tme finds emoji that can take (or have) skin tone modifiers
print(ext.count_tme("High five ✋🏽"))

# count_tones counts just the skin tone modifiers themselves
print(ext.count_tones("Waves 👋🏻 and 👋🏿"))

## 5. Version Selection

Emoji are added in each Unicode release. You can pin to a specific version
to match only emoji that existed at that point in time.

In [ ]:
ext_14 = Extractor(version='14.0')
ext_15 = Extractor(version='15.0')

# 🩷 Pink heart was introduced in Unicode 15.0
print(f"v14 detects 🩷: {ext_14.detect_emoji('🩷')}")   # False
print(f"v15 detects 🩷: {ext_15.detect_emoji('🩷')}")   # True

In [ ]:
ext_15_0 = Extractor(version='15.0')
ext_15_1 = Extractor(version='15.1')

# 🍋‍🟩 Lime is a ZWJ sequence introduced in 15.1
# In 15.0 it's counted as two separate emoji (lemon + green square)
# In 15.1 it's counted as one single emoji (lime)
print(f"v15.0 count: {ext_15_0.count_emoji('🍋‍🟩')}")
print(f"v15.1 count: {ext_15_1.count_emoji('🍋‍🟩')}")

## 6. Controlling Parallelism

By default, bulk methods (`count_all_emoji`, etc.) use up to 8 CPU cores.
You can configure this:

In [ ]:
# Use fewer workers
ext_limited = Extractor(n_workers=2)
print(f"Workers: {ext_limited.n_workers}")

# Disable multiprocessing entirely
ext_serial = Extractor(n_workers=1)
print(f"Workers: {ext_serial.n_workers}")

# Clean up worker processes when done
ext_limited.close()
ext_serial.close()

## 7. Performance

The trie-based engine is **27× faster** than the old regex engine for single strings,
and **115× faster** with multiprocessing for bulk datasets.

Single-line median latency: **~9µs** (compared to ~270µs with regex).

The `check_first` parameter from older versions is still accepted but has no effect —
the trie engine is already fast enough that pre-filtering adds no benefit.